In [1]:
!pip -q install requests beautifulsoup4

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [5]:
BASE_URL = "https://www.scrapethissite.com/pages/forms/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; DIS08-scraper/1.0)"}

def to_int(s, default=0):
    s = (s or "").strip()
    return default if s == "" else int(s)

def to_float(s, default=None):
    s = (s or "").strip()
    if s == "":
        return default
    return float(s)

def scrape_all_pages(sleep_seconds=0.5):
    all_rows = []
    page = 1

    while True:
        url = f"{BASE_URL}?page_num={page}"
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "html.parser")
        team_rows = soup.select("tr.team")
        if not team_rows:
            break

        for tr in team_rows:
            tds = tr.find_all("td")
            if len(tds) < 9:
                continue

            def txt(i):
                return tds[i].get_text(strip=True)

            all_rows.append({
                "Team Name": txt(0),
                "Year": to_int(txt(1)),
                "Wins": to_int(txt(2)),
                "Losses": to_int(txt(3)),
                "OT Losses": to_int(txt(4), default=0),     # <- FIX
                "Win %": to_float(txt(5), default=None),
                "Goals For (GF)": to_int(txt(6)),
                "Goals Against (GA)": to_int(txt(7)),
                "+ / -": to_int(txt(8)),
            })

        page += 1
        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)

df = scrape_all_pages()
df.shape, df.head()

((582, 9),
             Team Name  Year  Wins  Losses  OT Losses  Win %  Goals For (GF)  \
 0       Boston Bruins  1990    44      24          0  0.550             299   
 1      Buffalo Sabres  1990    31      30          0  0.388             292   
 2      Calgary Flames  1990    46      26          0  0.575             344   
 3  Chicago Blackhawks  1990    49      23          0  0.613             284   
 4   Detroit Red Wings  1990    34      38          0  0.425             273   
 
    Goals Against (GA)  + / -  
 0                 264     35  
 1                 278     14  
 2                 263     81  
 3                 211     73  
 4                 298    -25  )

In [7]:
df.to_csv("data.csv", index=False)
print("Saved data.csv with", len(df), "rows")

Saved data.csv with 582 rows


In [8]:
years = [1990, 2000, 2010]

subset = df[df["Year"].isin(years)].copy()

max_wins = subset.groupby("Year")["Wins"].max()

top_wins = (
    subset.merge(max_wins.rename("MaxWins"), on="Year")
          .query("Wins == MaxWins")
          [["Year", "Team Name", "Wins"]]
          .sort_values(["Year", "Team Name"])
)

top_wins

,Year,Team Name,Wins
3,1990,Chicago Blackhawks,49
29,2000,Colorado Avalanche,52
79,2010,Vancouver Canucks,54


In [9]:
years = [1991, 2001, 2011]

teams_per_year = (
    df[df["Year"].isin(years)]
      .groupby("Year")["Team Name"]
      .nunique()
      .reset_index(name="NumTeams")
      .sort_values("Year")
)

teams_per_year

,Year,NumTeams
0,1991,22
1,2001,30
2,2011,30
